In [29]:
# Hypothesis 2: Certain syntactic constructions occur often in explanations.
# 1. conditionals: when/if + (SUBJ/OBJ) + V
# 2. (DET and/or N) is OBJ
# 3. not X but Y
import spacy
nlp = spacy.load('en_core_web_sm')

In [30]:
sentence = 'When a person helps others, then they are helpful. This is not exactly rude, but kind.'

In [31]:
doc = nlp(sentence)

for token in doc:
    print(token.text, token.pos_, token.dep_, token.head.text)

When SCONJ advmod helps
a DET det person
person NOUN nsubj helps
helps VERB advcl are
others NOUN dobj helps
, PUNCT punct are
then ADV advmod are
they PRON nsubj are
are AUX ROOT are
helpful ADJ acomp are
. PUNCT punct are
This PRON nsubj is
is AUX ROOT is
not PART neg is
exactly ADV advmod rude
rude ADJ acomp is
, PUNCT punct rude
but CCONJ cc rude
kind ADJ conj rude
. PUNCT punct is


In [32]:
SUBJ_OBJ = {"nsubj", "nsubjpass", "csubj", "csubjpass", "expl", "dobj", "obj"}
SUBJ = ("nsubj", "nsubjpass", "expl")

In [33]:
#1. when/if

def has_conditional(doc):
    for tok in doc:
        if tok.lower_ in ("when", "if") and tok.dep_ in ("advmod", "mark"):
            verb = tok.head
            if verb.pos_ == "VERB" and tok.i < verb.i:
                # optional: grab the subject/object sitting between when/if and the verb
                args = [c for c in verb.children
                        if c.dep_ in SUBJ_OBJ and tok.i < c.i < verb.i]
                #print(f"Match: '{tok.text}' → verb '{verb.text}', subj/obj: {[a.text for a in args]}")
                return True
    return False

print(has_conditional(doc))

True


In [34]:
# 2. det / n IS

def has_be(doc):
    for tok in doc:
        if tok.lemma_ == "be" and tok.i > 0:
            j = tok.i - 1
            # skip modals/auxiliaries (can, will, has), negation (not, n't), adverbs (probably)
            while j >= 0 and doc[j].pos_ in ("AUX", "PART", "ADV"):
                j -= 1
            if j >= 0 and doc[j].pos_ in ("DET", "NOUN", "PROPN", "PRON"):
                return True
    return False

print(has_be(doc))

True


In [40]:
# 3. not x but y

def starts_new_clause(but):
    doc = but.doc
    if but.i + 1 >= len(doc):
        return False
    nxt = doc[but.i + 1]
    # "but I ..." or "but the dog ..." (determiner attached to a subject noun)
    return nxt.dep_ in SUBJ or (nxt.dep_ == "det" and nxt.head.dep_ in SUBJ)

def has_not_but(doc, max_gap=10):
    for sent in doc.sents:
        for tok in sent:
            if tok.lower_ not in ("not", "n't"):
                continue
            nxt = doc[tok.i + 1] if tok.i + 1 < sent.end else None
            if nxt is not None and nxt.lower_ in ("only", "just", "merely"):
                continue  # step 2
            window_end = min(tok.i + 1 + max_gap, sent.end)
            for later in doc[tok.i + 1 : window_end]:
                if later.lower_ == "but":
                    if starts_new_clause(later):  # step 3
                        break
                    return True
    return False

print(has_not_but(doc))

True


In [41]:
import pandas as pd

In [47]:
# go through all 1k explanations and create a frequency table:
# Conditional | presence in percentage of entries
# IS construction | presence in percentage of entries
# Not _ but _ | presence in percentage of entries

expl_df = pd.read_csv('eli5_comments.csv')
expls = [expl for expl in expl_df['explanation_1']]
len(expls)

1000

In [54]:
freq_table = pd.DataFrame(data={'Phenomenon': ['Conditionals', 'BE Construction', 'Not _ But _'],'Frequency (abs)': [0, 0, 0], 'Percentage': [0, 0, 0]})
freq_table

,Phenomenon,Frequency (abs),Percentage
0,Conditionals,0,0
1,BE Construction,0,0
2,Not _ But _,0,0


In [55]:
def calc_perc_1000(num):
  return round(num / 1000 * 100, 2)

In [56]:
for ex in expls:
  cond = has_conditional(nlp(ex))
  be = has_be(nlp(ex))
  not_but = has_not_but(nlp(ex))
  if cond: freq_table.at[0, 'Frequency (abs)'] += 1
  if be: freq_table.at[1, 'Frequency (abs)'] += 1
  if not_but: freq_table.at[2, 'Frequency (abs)'] += 1
  freq_table.at[0, 'Percentage'] = calc_perc_1000(freq_table.at[0, 'Frequency (abs)'])
  freq_table.at[1, 'Percentage'] = calc_perc_1000(freq_table.at[1, 'Frequency (abs)'])
  freq_table.at[2, 'Percentage'] = calc_perc_1000(freq_table.at[2, 'Frequency (abs)'])

freq_table


/tmp/ipykernel_3677/4106280052.py:9: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.1' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  freq_table.at[1, 'Percentage'] = calc_perc_1000(freq_table.at[1, 'Frequency (abs)'])


,Phenomenon,Frequency (abs),Percentage
0,Conditionals,454,45.4
1,BE Construction,907,90.7
2,Not _ But _,38,3.8
